# Pandas vs PySpark

✅ = syntax pandas bisa dipakai di Spark &nbsp;&nbsp; ❌ = tidak bisa, pakai versi Spark

In [ ]:
import os, sys

# Java dari env conda (supaya terbaca juga saat dijalankan di VS Code)
os.environ.setdefault("JAVA_HOME", os.path.join(sys.prefix, "Library" if os.name == "nt" else "lib/jvm"))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("PandasVsSpark") \
    .getOrCreate()

# Membaca Data

In [ ]:
# pandas: pd.read_csv(...)  ❌
orders = spark.read.csv('../data/superstore_orders.csv', header=True, inferSchema=True, escape='"')
orders.show(5)

In [ ]:
# pandas: df.shape  ❌
orders.count(), len(orders.columns)

In [ ]:
# pandas: df.columns  ✅
orders.columns

In [ ]:
# pandas: df.info()  ❌
orders.printSchema()

# Kenapa Harus show()

Spark itu **lazy**: tanpa `show()` hanya muncul struktur kolom, datanya belum dihitung.

In [ ]:
orders[orders['Sales'] > 3000]

In [ ]:
orders[orders['Sales'] > 3000].show()

In [ ]:
# tampil otomatis seperti pandas (hanya di Jupyter)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
orders

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", False)

# Ganti Nama Kolom

In [ ]:
# pandas: df.columns = [...]  ❌
def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(orders)
orders.columns

# Pilih, Filter, Urutkan

In [ ]:
# pandas: df[['order_id', 'sales']]  ✅
orders[['order_id', 'region', 'sales']].show(5)

In [ ]:
# pandas: df[df['sales'] > 3000]  ✅
orders[orders['sales'] > 3000].show()

In [ ]:
# pandas: df[(...) & (...)]  ✅
orders[(orders['sales'] > 1000) & (orders['region'] == 'West')].show(5)

In [ ]:
# pandas: df.query('sales > 3000')  ❌
orders.filter('sales > 3000').count()

In [ ]:
# pandas: df.sort_values('sales', ascending=False)  ❌
orders.sort('sales', ascending=False).show(5)

In [ ]:
# pandas: df['region'].value_counts()  ❌
orders.groupby('region').count().show()

In [ ]:
# pandas: df['region'].unique()  ❌
orders[['region']].distinct().show()

# Kolom Baru

In [ ]:
# pandas: df['unit_price'] = df['sales'] / df['quantity']  ❌
orders.withColumn('unit_price', F.round(orders['sales'] / orders['quantity'], 2)).show(5)

# Membersihkan Data

In [ ]:
# duplikat. pandas: df.drop_duplicates()  ✅
orders.count(), orders.drop_duplicates().count()

In [ ]:
# 1 baris per ID. pandas: df.drop_duplicates(subset=[...])  ✅
products = to_snake_case(spark.read.csv('../data/superstore_products.csv', header=True, inferSchema=True, escape='"'))
products.count(), products.drop_duplicates(subset=['product_id']).count()

In [ ]:
# nilai kosong. pandas: df.isna().sum()  ❌
for c in orders.columns:
    print(c, orders[orders[c].isNull()].count())

In [ ]:
# pandas: df.fillna({...}) / df.dropna(subset=[...])  ✅
orders.dropna(subset=['sales']).count()

In [ ]:
# teks. pandas: df['region'].str.strip().str.title()  ❌
orders.withColumn('region', F.initcap(F.trim('region')))[['region']].distinct().show()

In [ ]:
# angka 0 di depan. pandas: .astype(str).str.zfill(5)  ❌
orders.withColumn('postal_code', F.lpad(orders['postal_code'].astype('string'), 5, '0'))[['city', 'postal_code']].show(5)

In [ ]:
# tanggal. pandas: pd.to_datetime(...), .dt.year  ❌
dated = orders.withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))

dated.select('order_date',
             F.year('order_date').alias('year'),
             F.quarter('order_date').alias('quarter'),
             F.month('order_date').alias('month'),
             F.date_format('order_date', 'MMMM').alias('month_name'),
             F.date_format('order_date', 'yyyyMMdd').cast('int').alias('date_id')).show(5)

# Group By

In [ ]:
# pandas: df.groupby('region').agg({'sales': 'sum'})  ✅
orders.groupby('region').agg({'sales': 'sum'}).show()

In [ ]:
orders.groupby('region').agg(
    F.round(F.sum('sales'), 2).alias('total_sales'),
    F.countDistinct('customer_id').alias('customers'),
).show()

# Join

In [ ]:
# pandas: pd.merge(orders, products, on='product_id', how='left')  ❌
customers = to_snake_case(spark.read.csv('../data/superstore_customers.csv', header=True, inferSchema=True, escape='"'))

orders.join(products, on='product_id', how='left').show(5)

In [ ]:
# join 3 tabel: cek jumlah baris (bertambah karena ID duplikat)
full = orders.join(customers, on='customer_id', how='left').join(products, on='product_id', how='left')
orders.count(), full.count()

In [ ]:
# order yang produknya tidak ada di file products
orders.join(products, on='product_id', how='left_anti').show()

# Union & Nomor Urut

In [ ]:
# pandas: pd.concat([a, b])  ❌
order_dates = dated.select(F.col('order_date').alias('full_date'))
ship_dates = orders.select(F.to_date('ship_date', 'M/d/yyyy').alias('full_date'))
order_dates.union(ship_dates).distinct().count()

In [ ]:
# nomor urut 1, 2, 3, ...
cities = orders[['state', 'city']].drop_duplicates()
cities.withColumn('city_id', F.row_number().over(Window.orderBy('state', 'city'))).show(5)

In [ ]:
spark.stop()